In [1]:
from anthropic import Anthropic
from dotenv import load_dotenv
import os

In [2]:
load_dotenv(override=True)

client = Anthropic()
model = "claude-sonnet-5"

In [3]:
coordinator_agent = {
    "name": "shopassist_coordinator",
    "description": "Routes complex requests and writes the reply.",
    "allowedTools": ["Task", "get_customer", "escalate_to_human"],
    "system": """Break complex requests into meaningful tasks.  Use
    subagents for investigation.  All communication with the customer
    goes through you."""
}

In [4]:
billing_analyst_agent = {
    "name": "billing_analyst_agent",
    "description": "",
    "allowedTools": ["get_payment_events", "get_redund_status"],
    "system": """Analyze billing facts only.  Do not contact the customer.
    Return all findings to the coordinator_agent."""
}

In [5]:
order_investigation_agent = {
    "name": "order_investigation_agent",
    "description": "",
    "allowedTools": ["lookup_order", "get_shipment_events"],
    "system": """Lookup orders and report what happened, including ship date, 
    if the order was delivered, and delivery date if it was delivered.  
    Do not contact the customer.  Return all findings to the coordinator_agent."""
}

In [6]:
policy_review_agent = {
    "name": "policy_review_agent",
    "description": "",
    "allowedTools": ["search_policy"],
    "system": """Look up the rules and policies, review exception conditions, 
    and return confidence score.  Do not contact the customer.  Return all 
    findings to the coordinator_agent."""
}

In [8]:
billing_task = {
    "tool": "Task", "subagent": "billing_analysis_agent",
    "prompt": """Investigate a possible duplicate charge.
    Customer context: customer_id C-8821 - verified: true
    Original message: '...charged twice for A1042...'
    Relevant issue: possible duplicate charge for A1042
    Return: { issue, evidence[], conculusion, 
    recommended_next_step, missing_information[] }"""
}

In [10]:
policy_task = """Review return eligibility.
CONTENT: can they still return a charger from a past order?
METADATA: customer_id C-8821 - order A0977 - item charger
    purchase_date 2026-05-12 - current_date 2026-06-20
PRIOR FINDINGS: verified custoer - no redund issued yet
Return: rule - eligibility - missing facts - human review?"""

In [12]:
def refund_workflow(customer_id: str, order_id: str, message: str):
    customer = get_customer(customer_id)
    order = lookup_order(
        customer_id=customer["id"],
        order_id=order_id
    )
    policy = check_return_policy(
        customer_id=customer["id"],
        order_id=order["id"]
    )
    decision = decide_refund(
        customer=customer,
        order=order,
        policy=policy,
        customer_message=message
    )
    return draft_customer_response(decision)

In [ ]:
classification = classify_customer_issue(message)
extraction = extract_return_request(message)

order = lookup_order(
    customer_id=session.customer_id,
    order_id=extraction["order_id"]
)
policy = check_return_policy(
    customer_id=session.customer_id,
    order_id=order["id"],
    item=extraction["item"],
    reason=extraction["reason"]
)
response = draft_customer_response(
    classification=classification,
    extraction=extraction,
    order=order,
    policy=policy
)